In [ ]:
!pip install nltk scikit-learn textblob wordcloud tensorflow matplotlib

In [ ]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist

# Q1: Unique paragraph (5-6 sentences) about Technology
paragraph = """Artificial intelligence is transforming modern technology at an incredible pace! 
Machine learning algorithms help doctors diagnose diseases faster and more accurately than ever before. 
Meanwhile, self-driving cars use computer vision and sensors to navigate busy city streets safely. 
Don't you think cloud computing and edge devices make data processing seamlessly connected? 
However, ethical challenges like data privacy and algorithmic bias still require careful regulation. 
Ultimately, technology continues to reshape how humans live, work, and communicate every single day."""

# 1. Convert text to lowercase and remove punctuation using re
clean_text = re.sub(r'[^\w\s]', '', paragraph.lower())
print("1. Cleaned Text (Lowercase & No Punctuation):\n", clean_text, "\n")

# 2. Tokenize the text into sentences and words
sentences = sent_tokenize(paragraph)
words = word_tokenize(clean_text)
print(f"2. Sentence Tokens ({len(sentences)} sentences):\n", sentences, "\n")
print(f"   Word Tokens (from cleaned text, {len(words)} words):\n", words, "\n")

# 3. Compare Python's split() vs NLTK's word_tokenize() on the original paragraph
split_words = paragraph.split()
nltk_words = word_tokenize(paragraph)
print("3. Comparison on raw text:")
print("   Python split() output:       ", split_words[:15])
print("   NLTK word_tokenize() output: ", nltk_words[:15])
print("""   Difference Explanation:
   - Python's .split() simply splits by whitespace, keeping punctuation attached to words (e.g., 'pace!', "Don't").
   - NLTK's word_tokenize() uses grammatical rules to separate punctuation (e.g., 'pace', '!') and splits contractions (e.g., 'Do', "n't").\n""")

# 4. Remove stopwords (using NLTK's stopwords list)
stop_words = set(stopwords.words('english'))
filtered_words = [w for w in words if w not in stop_words]
print("4. Words after Stopword Removal:\n", filtered_words, "\n")

# 5. Display word frequency distribution (excluding stopwords)
fdist = FreqDist(filtered_words)
print("5. Top 10 Most Frequent Words (excluding stopwords):")
for word, freq in fdist.most_common(10):
    print(f"   {word}: {freq}")

# Optional: Plot the frequency distribution
plt.figure(figsize=(10, 4))
fdist.plot(15, title="Word Frequency Distribution (Excluding Stopwords)")
plt.show()

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

# 1. Extract all words with only alphabets using re.findall()
alpha_words = re.findall(r'\b[a-zA-Z]+\b', paragraph.lower())
print("1. Alphabetic Words Only:\n", alpha_words, "\n")

# 2. Remove stop words using NLTK's stopword list
filtered_alpha = [w for w in alpha_words if w not in stop_words]
print("2. Filtered Alphabetic Words (No Stopwords):\n", filtered_alpha, "\n")

# 3. Perform stemming with PorterStemmer
ps = PorterStemmer()
stemmed_words = [ps.stem(w) for w in filtered_alpha]

# 4. Perform lemmatization with WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(w) for w in filtered_alpha]

# 5. Compare the stemmed and lemmatized outputs
comparison_df = pd.DataFrame({
    'Original Word': filtered_alpha,
    'Stemmed (Porter)': stemmed_words,
    'Lemmatized (WordNet)': lemmatized_words
}).drop_duplicates()

print("3, 4 & 5. Comparison of Original, Stemmed, and Lemmatized Words:")
print(comparison_df.to_string(index=False))

print("""\nExplanation — When to prefer one over the other:
- Stemming (PorterStemmer) chops off word suffixes using heuristic rules (e.g., 'technology' -> 'technolog', 'accurately' -> 'accur'). Prefer Stemming when speed is critical and large-scale search/indexing (like Information Retrieval) doesn't require valid dictionary words.
- Lemmatization (WordNetLemmatizer) uses vocabulary and morphological analysis to return valid base dictionary words (lemmas). Prefer Lemmatization for tasks where meaning and context matter, such as Question Answering, Chatbots, and Sentiment Analysis.""")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Q3: Choose 3 short texts (Product Reviews / Headlines)
texts = [
    "The new smartphone battery life is amazing and the camera quality is superb.",
    "The laptop performance is fast, but the battery drains very quickly during gaming.",
    "Customer service was helpful and resolved my smartphone camera issue quickly."
]

# 1. Use CountVectorizer to generate the Bag of Words representation
count_vec = CountVectorizer(stop_words='english')
bow_matrix = count_vec.fit_transform(texts)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=count_vec.get_feature_names_out(), index=['Text 1', 'Text 2', 'Text 3'])
print("1. Bag of Words (BoW) Representation:\n")
print(bow_df, "\n")

# 2. Use TfidfVectorizer to compute TF-IDF scores
tfidf_vec = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vec.fit_transform(texts)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vec.get_feature_names_out(), index=['Text 1', 'Text 2', 'Text 3'])
print("2. TF-IDF Score Matrix (Rounded to 3 decimal places):\n")
print(tfidf_df.round(3), "\n")

# 3. Print and interpret the top 3 keywords from each text using TF-IDF
feature_names = np.array(tfidf_vec.get_feature_names_out())
print("3. Top 3 Keywords per Text using TF-IDF:")
for i, row in enumerate(tfidf_matrix.toarray()):
    top3_indices = row.argsort()[-3:][::-1]
    top3_words = [(feature_names[idx], round(row[idx], 3)) for idx in top3_indices]
    print(f"   Text {i+1}: {top3_words}")

print("""\nInterpretation:
- TF-IDF assigns higher weights to words that are unique to a specific document (e.g., 'amazing', 'superb', 'life' in Text 1; 'laptop', 'gaming', 'drains' in Text 2) while down-weighting words that appear across multiple documents (such as 'battery', 'smartphone', 'camera', and 'quickly').""")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Q4: Write 2 short texts (4-6 lines each) describing two different technologies (AI vs Blockchain)
tech_ai = """Artificial Intelligence enables machines to learn from data and make intelligent decisions.
Deep learning and neural networks power applications like natural language processing and computer vision.
Modern AI systems process massive datasets to automate complex digital workflows and predictive analytics.
It relies heavily on high-performance computing, cloud infrastructure, and advanced algorithms.
AI is widely adopted in healthcare, finance, and autonomous systems to improve efficiency."""

tech_blockchain = """Blockchain is a decentralized digital ledger technology that records transactions securely across a network.
It uses cryptographic hashing and consensus algorithms to ensure data integrity and immutability.
Smart contracts on blockchain platforms automate digital workflows without requiring central intermediaries.
This technology powers cryptocurrencies and secure supply chain management systems globally.
Blockchain is widely adopted in finance, banking, and cybersecurity to enhance transparency."""

# 1. Preprocess and tokenize both texts
def preprocess_and_tokenize(text):
    text_clean = re.sub(r'[^\w\s]', '', text.lower())
    tokens = word_tokenize(text_clean)
    return [w for w in tokens if w not in stop_words]

tokens_ai = preprocess_and_tokenize(tech_ai)
tokens_blockchain = preprocess_and_tokenize(tech_blockchain)

print("1. Preprocessed Tokens (AI):\n", tokens_ai[:15], "...\n")
print("   Preprocessed Tokens (Blockchain):\n", tokens_blockchain[:15], "...\n")

# 2a. Calculate Jaccard Similarity using sets
set_ai = set(tokens_ai)
set_bc = set(tokens_blockchain)
intersection = set_ai.intersection(set_bc)
union = set_ai.union(set_bc)
jaccard_sim = len(intersection) / len(union)

print(f"2a. Jaccard Similarity: {jaccard_sim:.4f}")
print(f"    Shared Words: {intersection}\n")

# 2b. Calculate Cosine Similarity using TfidfVectorizer + cosine_similarity()
vec = TfidfVectorizer(stop_words='english')
tfidf_vecs = vec.fit_transform([tech_ai, tech_blockchain])
cos_sim = cosine_similarity(tfidf_vecs[0:1], tfidf_vecs[1:2])[0][0]

print(f"2b. Cosine Similarity (TF-IDF): {cos_sim:.4f}\n")

# 2c. Analyze which similarity metric gives better insights
print("""2c. Analysis of Similarity Metrics:
- Jaccard Similarity only checks binary word overlap (presence/absence in sets) and ignores word frequency and term importance.
- Cosine Similarity with TF-IDF accounts for term frequency and down-weights common terms while capturing vector magnitude. Therefore, Cosine Similarity provides better insights for multi-line documents because it measures directional alignment in feature space rather than simple set overlap.""")

In [ ]:
from textblob import TextBlob
from wordcloud import WordCloud

# Q5: Product/Service Reviews
reviews = [
    "This wireless headphone has crystal clear sound and super comfortable ear pads! Best purchase ever.",
    "The delivery was terribly slow and the package arrived damaged. Very disappointed with the service.",
    "The product works as described in the manual. It is standard and does the basic job.",
    "Amazing customer support and fantastic build quality! I highly recommend this brand to everyone.",
    "Battery life is awful and the connection keeps dropping every five minutes."
]

# 1 & 2. Find polarity & subjectivity using TextBlob and classify into Positive / Negative / Neutral
results = []
positive_reviews = []

for rev in reviews:
    blob = TextBlob(rev)
    polarity = round(blob.sentiment.polarity, 3)
    subjectivity = round(blob.sentiment.subjectivity, 3)
    
    if polarity > 0.1:
        sentiment = "Positive"
        positive_reviews.append(rev)
    elif polarity < -0.1:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"
        
    results.append({
        'Review': rev[:55] + "...",
        'Polarity': polarity,
        'Subjectivity': subjectivity,
        'Classification': sentiment
    })

sentiment_df = pd.DataFrame(results)
print("1 & 2. Sentiment Analysis Results (TextBlob):\n")
print(sentiment_df.to_string(index=False), "\n")

# 3. Create a word cloud using the wordcloud library for all positive reviews
positive_text = " ".join(positive_reviews)
wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='viridis').generate(positive_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud for Positive Reviews", fontsize=14)
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

# Q6: Choose a ~100-word paragraph as training data
training_data = """Space exploration pushes the boundaries of human knowledge and scientific discovery.
Astronomers use powerful telescopes to observe distant galaxies and nebulae across the universe.
Rockets carry satellites and robotic rovers to explore planets like Mars and Venus.
Future missions aim to build sustainable lunar bases for astronauts on the Moon.
Deep space probes travel beyond our solar system to study cosmic radiation and interstellar space.
Every mission brings new technology that benefits life on Earth and inspires future generations.
Scientists continue searching for signs of water and microbial life on distant exoplanets."""

# 1. Tokenize text using Tokenizer()
tokenizer = Tokenizer()
corpus = [line.strip() for line in training_data.lower().split('\n') if line.strip()]
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1
print(f"1. Total Unique Words in Vocabulary: {total_words - 1}")

# 2. Create input sequences and build a simple LSTM model
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

X = input_sequences[:, :-1]
labels = input_sequences[:, -1]
y = to_categorical(labels, num_classes=total_words)

# Build the LSTM Model
model = Sequential([
    Embedding(total_words, 32, input_length=max_sequence_len - 1),
    LSTM(64),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print("\n2. Model Summary:")
model.summary()

# 3. Train the model and generate 2-3 new lines of text starting from seed words
print("\n3. Training the model...")
model.fit(X, y, epochs=150, verbose=0)
print("   Training complete!\n")

def generate_text(seed_text, next_words=8):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]
        
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text.capitalize() + "."

# Generate 3 new lines from seed words
seed_words = ["Space exploration", "Astronomers use", "Future missions"]
print("Generated Lines of Text:")
for seed in seed_words:
    print(" ->", generate_text(seed, next_words=8))